# 경기·인천 경쟁권 25km 반영 네트워크 접근성 분석

- 목적: 서울 단독 분석을 보존하고, 외부 25km 경쟁권을 반영한 접근 가능 격자-가맹점 pair를 생성
- 범위: 서울 100m 격자 + 경기·인천 외부 25km 100m 격자, 서울 가맹점 + 경기·인천 가맹점
- 저장: 접근성 분석에 필요한 최소 테이블만 저장


## 1. 경로 및 패키지 설정

- 기존 05번 네트워크 분석과 동일한 패키지 사용
- 대용량 중복 저장 방지를 위해 graphml/node/edge 원본 산출물은 저장하지 않음
- 접근성 계산에 필요한 pair 테이블과 node 연결테이블만 압축 parquet로 저장


In [ ]:
import json
import pathlib
import time
from collections import defaultdict

import geopandas as gpd
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from shapely.geometry import LineString

pd.set_option("display.max_columns", 80)

BASE_PATH = pathlib.Path().resolve()

if BASE_PATH.name == "notebooks":
    BASE_PATH = BASE_PATH.parent.parent
elif BASE_PATH.name == "analysis_table":
    BASE_PATH = BASE_PATH.parent
elif BASE_PATH.name != "oracle_mnc_project" and (BASE_PATH / "analysis_table").exists():
    BASE_PATH = BASE_PATH

ANALYSIS_PATH = BASE_PATH / "analysis_table"
DATA_PATH = ANALYSIS_PATH / "data"
INPUT_PATH = DATA_PATH / "input"
OUTPUT_PATH = DATA_PATH / "output"
RAW_PATH = BASE_PATH / "data" / "raw"
SPATIAL_PATH = RAW_PATH / "spatial"

NETWORK_INPUT_PATH = INPUT_PATH / "network"
NETWORK_OUTPUT_PATH = OUTPUT_PATH / "network_competition_25km"
NETWORK_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

ktdb_year = 2025

KTDB_PATH = (
    NETWORK_INPUT_PATH / "ktdb_transport_network" /
    "2024-TRNT-AG-01 (도로철도통합)수도권 네트워크(2023-2035)" /
    f"네트워크_{ktdb_year}"
)

NODE_PATH = KTDB_PATH / "01. Node_Link data" / f"{ktdb_year}node.txt"
LINK_PATH = KTDB_PATH / "01. Node_Link data" / f"{ktdb_year}link.txt"
KTDB_TRANSIT_DIR = KTDB_PATH / "03. Transit"
TRANSIT_INFO_PATH = KTDB_TRANSIT_DIR / f"TransitInfo_{ktdb_year}.txt"
TRANSIT_ROUTE_PATH = KTDB_TRANSIT_DIR / f"TransitPath_{ktdb_year}.txt"

print("BASE_PATH:", BASE_PATH)
print("OUTPUT:", NETWORK_OUTPUT_PATH)
print("KTDB_PATH exists:", KTDB_PATH.exists())


## 2. 분석 입력 데이터 불러오기

- 서울 단독 결과는 수정하지 않고 확장 분석용 테이블을 새로 생성
- 격자와 가맹점은 서울/외부권역 구분 칼럼을 유지
- 숙박·여행사·교통수단은 접근성 분석 대상에서 제외


In [ ]:
# 격자 데이터
seoul_grid = gpd.read_file(OUTPUT_PATH / "서울시_100m_추정인구.gpkg")
external_grid = gpd.read_file(OUTPUT_PATH / "인천경기_외부25km_100m_추정인구.gpkg")

seoul_grid = seoul_grid.copy()
external_grid = external_grid.copy()

seoul_grid["시도"] = "서울특별시"
seoul_grid["서울여부"] = True
seoul_grid["외부25km권역여부"] = False
seoul_grid["계산대상_500m확장여부"] = False

external_grid["서울여부"] = False

grid_cols = [
    "GRID_CD",
    "시도",
    "서울여부",
    "행정동코드",
    "시군구",
    "행정동",
    "중심점_x",
    "중심점_y",
    "추정_인구수",
    "GRID_CD_500",
    "외부25km권역여부",
    "계산대상_500m확장여부",
    "geometry",
]

for col in grid_cols:
    if col not in seoul_grid.columns:
        seoul_grid[col] = np.nan
    if col not in external_grid.columns:
        external_grid[col] = np.nan

grid = pd.concat(
    [seoul_grid[grid_cols], external_grid[grid_cols]],
    ignore_index=True
)
grid = gpd.GeoDataFrame(grid, geometry="geometry", crs=seoul_grid.crs).to_crs("EPSG:5179")

print("격자 구조:", grid.shape)
print("격자 CRS:", grid.crs)
print("GRID_CD 중복:", grid["GRID_CD"].duplicated().sum())
print("격자 결측")
print(grid[["GRID_CD", "시도", "시군구", "행정동", "중심점_x", "중심점_y"]].isna().sum())
print("시도별 격자 수")
print(grid["시도"].value_counts(dropna=False))

# 가맹점 데이터
seoul_store = gpd.read_file(OUTPUT_PATH / "서울시_문화누리카드_가맹점_2026.gpkg")
external_store = gpd.read_file(OUTPUT_PATH / "인천경기_문화누리카드_가맹점_2026.gpkg")

seoul_store = seoul_store.copy()
external_store = external_store.copy()

seoul_store["서울여부"] = True
if "시도" not in seoul_store.columns:
    seoul_store["시도"] = "서울특별시"
external_store["서울여부"] = False

store_cols = [
    "가맹점_ID",
    "가맹점명",
    "서울여부",
    "시도",
    "시군구",
    "주소",
    "위도",
    "경도",
    "대분류",
    "중분류",
    "소분류",
    "전화결제",
    "장애인친화시설",
    "찾아가는문화서비스",
    "노령인구_편의서비스",
    "geometry",
]

for col in store_cols:
    if col not in seoul_store.columns:
        seoul_store[col] = np.nan
    if col not in external_store.columns:
        external_store[col] = np.nan

store = pd.concat(
    [seoul_store[store_cols], external_store[store_cols]],
    ignore_index=True
)
store = gpd.GeoDataFrame(store, geometry="geometry", crs=seoul_store.crs).to_crs("EPSG:5179")

exclude_columns = ["숙박", "여행사", "교통수단"]
store = store[~store["중분류"].isin(exclude_columns)].copy()

print("\n가맹점 구조:", store.shape)
print("가맹점 CRS:", store.crs)
print("가맹점_ID 중복:", store["가맹점_ID"].duplicated().sum())
print("가맹점 결측")
print(store[["가맹점_ID", "시도", "시군구", "중분류", "geometry"]].isna().sum())
print("중분류별 가맹점 수")
print(store["중분류"].value_counts())

# 분석용 차원 테이블 저장
grid_dim_cols = [
    "GRID_CD", "시도", "서울여부", "행정동코드", "시군구", "행정동",
    "중심점_x", "중심점_y", "추정_인구수", "GRID_CD_500",
    "외부25km권역여부", "계산대상_500m확장여부", "geometry"
]

store_dim_cols = [
    "가맹점_ID", "가맹점명", "서울여부", "시도", "시군구", "주소",
    "위도", "경도", "대분류", "중분류", "소분류",
    "전화결제", "장애인친화시설", "찾아가는문화서비스",
    "노령인구_편의서비스", "geometry"
]

grid[grid_dim_cols].to_parquet(
    NETWORK_OUTPUT_PATH / "경쟁권25km_분석격자.parquet",
    index=False,
    compression="zstd"
)


## 3. 분석권역 생성

- 서울 행정동 외곽경계를 기준으로 25km 경쟁권을 설정
- 도보 그래프는 도보 750m 기준에 맞춰 서울 경계 2km buffer 내부만 수집
- 대중교통 그래프는 경쟁권에 2km를 추가하여 경계부 네트워크 단절을 완화


In [ ]:
seoul_bound = gpd.read_file(OUTPUT_PATH / "서울시_시군구_행정동_경계.gpkg")
seoul_bound = seoul_bound.to_crs("EPSG:5179").dissolve()[["geometry"]].reset_index(drop=True)

competition_area_5179 = gpd.GeoDataFrame(
    geometry=seoul_bound.geometry.buffer(25_000),
    crs="EPSG:5179"
)

walk_area_5179 = gpd.GeoDataFrame(
    geometry=seoul_bound.geometry.buffer(2_000),
    crs="EPSG:5179"
)

transit_area_5179 = gpd.GeoDataFrame(
    geometry=competition_area_5179.geometry.buffer(2_000),
    crs="EPSG:5179"
)

print("서울 경계 면적(km2):", seoul_bound.area.sum() / 1_000_000)
print("25km 경쟁권 면적(km2):", competition_area_5179.area.sum() / 1_000_000)
print("도보 그래프 수집 면적(km2):", walk_area_5179.area.sum() / 1_000_000)
print("대중교통 필터 면적(km2):", transit_area_5179.area.sum() / 1_000_000)

competition_polygon = competition_area_5179.geometry.iloc[0]
before_store_count = len(store)
store = store[store.geometry.intersects(competition_polygon)].copy()

print("25km 경쟁권 밖 제외 가맹점 수:", before_store_count - len(store))
print("25km 경쟁권 내부 분석 가맹점 수:", len(store))
print("경쟁권 내부 중분류별 가맹점 수")
print(store["중분류"].value_counts())

store[store_dim_cols].to_parquet(
    NETWORK_OUTPUT_PATH / "경쟁권25km_분석가맹점.parquet",
    index=False,
    compression="zstd"
)

competition_area_5179.to_file(
    NETWORK_OUTPUT_PATH / "경쟁권25km_분석권역.gpkg",
    driver="GPKG"
)


## 4. 도보 네트워크 구축 및 snapping

- 기존 서울 분석과 동일하게 OSMnx 보행 네트워크 사용
- 격자 중심점과 가맹점 POI를 최근접 도보 node에 연결
- 격자 스냅거리는 100m 격자 내부 위치 오차를 반영해 35.35m 차감


In [ ]:
ox.settings.use_cache = True
ox.settings.timeout = 300

walk_polygon_4326 = walk_area_5179.to_crs("EPSG:4326").geometry.iloc[0]

start_time = time.time()
walk_graph = ox.graph_from_polygon(
    walk_polygon_4326,
    network_type="walk",
    simplify=True,
    retain_all=False
)
walk_graph = ox.project_graph(walk_graph, to_crs="EPSG:5179")

walk_node = ox.graph_to_gdfs(walk_graph, edges=False)

print("도보 그래프 node 수:", walk_graph.number_of_nodes())
print("도보 그래프 edge 수:", walk_graph.number_of_edges())
print("도보 그래프 구축 시간(분):", round((time.time() - start_time) / 60, 2))


In [ ]:
walk_area_polygon = walk_area_5179.geometry.iloc[0]

walk_grid = grid[grid.geometry.intersects(walk_area_polygon)].copy()
walk_grid["중심점"] = gpd.points_from_xy(
    walk_grid["중심점_x"],
    walk_grid["중심점_y"],
    crs="EPSG:5179"
)

walk_store = store[store.geometry.intersects(walk_area_polygon)].copy()

print("도보 분석 대상 격자 수:", len(walk_grid), "/", len(grid))
print("도보 분석 대상 가맹점 수:", len(walk_store), "/", len(store))

walk_grid["도보_노드ID"] = ox.distance.nearest_nodes(
    walk_graph,
    walk_grid["중심점_x"],
    walk_grid["중심점_y"]
)

walk_store["도보_노드ID"] = ox.distance.nearest_nodes(
    walk_graph,
    walk_store.geometry.x,
    walk_store.geometry.y
)

snap_node = walk_node[["geometry"]].copy().rename(columns={"geometry": "노드_geometry"})

walk_grid_snap = walk_grid[
    ["GRID_CD", "시도", "시군구", "행정동", "중심점", "도보_노드ID"]
].merge(
    snap_node,
    left_on="도보_노드ID",
    right_index=True,
    how="left"
)

walk_grid_snap["도보_스냅거리"] = (
    walk_grid_snap["중심점"].distance(walk_grid_snap["노드_geometry"])
)

walk_store_snap = walk_store[
    ["가맹점_ID", "가맹점명", "시도", "시군구", "중분류", "소분류", "geometry", "도보_노드ID"]
].merge(
    snap_node,
    left_on="도보_노드ID",
    right_index=True,
    how="left"
)

walk_store_snap["도보_스냅거리"] = (
    walk_store_snap.geometry.distance(walk_store_snap["노드_geometry"])
)

grid_snap_offset = (70.7 + 0) / 2
walk_grid_snap["격자_스냅거리_보정"] = (
    walk_grid_snap["도보_스냅거리"] - grid_snap_offset
).clip(lower=0)

print("격자 도보 node 결측:", walk_grid_snap["도보_노드ID"].isna().sum())
print("가맹점 도보 node 결측:", walk_store_snap["도보_노드ID"].isna().sum())
print("격자 도보 스냅거리 통계")
print(walk_grid_snap["도보_스냅거리"].describe())
print("가맹점 도보 스냅거리 통계")
print(walk_store_snap["도보_스냅거리"].describe())
print("격자 스냅거리 150m 초과:", (walk_grid_snap["도보_스냅거리"] > 150).sum())
print("가맹점 스냅거리 100m 초과:", (walk_store_snap["도보_스냅거리"] > 100).sum())

walk_grid_snap[
    ["GRID_CD", "도보_노드ID", "도보_스냅거리", "격자_스냅거리_보정"]
].to_parquet(
    NETWORK_OUTPUT_PATH / "경쟁권25km_격자_도보노드_연결테이블.parquet",
    index=False,
    compression="zstd"
)

walk_store_snap[
    ["가맹점_ID", "도보_노드ID", "도보_스냅거리"]
].to_parquet(
    NETWORK_OUTPUT_PATH / "경쟁권25km_가맹점_도보노드_연결테이블.parquet",
    index=False,
    compression="zstd"
)


## 5. 도보 접근 가능 pair 생성

- 수식: `접근비용 = max(격자_스냅거리 - 35.35m, 0) + 도보_네트워크거리 + 가맹점_스냅거리`
- 기준: 마을단위시설 도보 약 10분·750m
- 실행: 가맹점 node 기준 reverse Dijkstra로 750m 이내 도달 가능한 격자 탐색


In [ ]:
walk_columns = ["도서", "문화체험", "음악", "영상", "체육시설", "체육용품"]
walk_limit = 750

walk_store_target = walk_store_snap[walk_store_snap["중분류"].isin(walk_columns)].copy()

node_grid_dict = defaultdict(list)
for idx, row in walk_grid_snap.iterrows():
    node_grid_dict[row["도보_노드ID"]].append(idx)

node_store_dict = defaultdict(list)
for idx, row in walk_store_target.iterrows():
    node_store_dict[row["도보_노드ID"]].append(idx)

grid_info = walk_grid_snap.to_dict("index")
store_info = walk_store_target.to_dict("index")

print("도보 대상 가맹점 수:", len(walk_store_target))
print("도보 대상 가맹점 node 수:", len(node_store_dict))
print("격자가 붙은 도보 node 수:", len(node_grid_dict))
print(walk_store_target["중분류"].value_counts())

walk_pair_path = NETWORK_OUTPUT_PATH / "경쟁권25km_도보접근성.parquet"
if walk_pair_path.exists():
    walk_pair_path.unlink()

pair_schema = pa.schema([
    ("GRID_CD", pa.string()),
    ("가맹점_ID", pa.string()),
    ("접근수단", pa.string()),
    ("접근비용", pa.float32()),
])

writer = pq.ParquetWriter(walk_pair_path, pair_schema, compression="zstd")

walk_graph_reverse = walk_graph.reverse(copy=False)
min_grid_snap = walk_grid_snap["격자_스냅거리_보정"].min()

batch_rows = []
batch_size = 500_000
pair_count = 0
start_time = time.time()
total_store_node_count = len(node_store_dict)

try:
    for n, (store_node, store_idx_list) in enumerate(node_store_dict.items(), start=1):
        if store_node not in walk_graph_reverse:
            continue

        min_store_snap = min(store_info[store_idx]["도보_스냅거리"] for store_idx in store_idx_list)
        network_cutoff = walk_limit - min_grid_snap - min_store_snap
        if network_cutoff <= 0:
            continue

        lengths = nx.single_source_dijkstra_path_length(
            walk_graph_reverse,
            store_node,
            cutoff=network_cutoff,
            weight="length"
        )

        for grid_node, network_dist in lengths.items():
            grid_idx_list = node_grid_dict.get(grid_node, [])
            if len(grid_idx_list) == 0:
                continue

            for grid_idx in grid_idx_list:
                grid_row = grid_info[grid_idx]
                grid_snap_corrected = grid_row["격자_스냅거리_보정"]

                for store_idx in store_idx_list:
                    store_row = store_info[store_idx]
                    final_dist = (
                        grid_snap_corrected
                        + network_dist
                        + store_row["도보_스냅거리"]
                    )

                    if final_dist <= walk_limit:
                        batch_rows.append((
                            str(grid_row["GRID_CD"]),
                            str(store_row["가맹점_ID"]),
                            "도보",
                            float(final_dist),
                        ))

        if len(batch_rows) >= batch_size:
            batch_df = pd.DataFrame(
                batch_rows,
                columns=["GRID_CD", "가맹점_ID", "접근수단", "접근비용"]
            )
            batch_df["접근비용"] = batch_df["접근비용"].astype("float32")
            writer.write_table(pa.Table.from_pandas(batch_df, schema=pair_schema, preserve_index=False))
            pair_count += len(batch_df)
            batch_rows = []

        if n % 200 == 0:
            elapsed_min = (time.time() - start_time) / 60
            print(f"{n:,} / {total_store_node_count:,}개 도보 가맹점 node 처리 | pair {pair_count:,}+ | {elapsed_min:.1f}분")

    if len(batch_rows) > 0:
        batch_df = pd.DataFrame(
            batch_rows,
            columns=["GRID_CD", "가맹점_ID", "접근수단", "접근비용"]
        )
        batch_df["접근비용"] = batch_df["접근비용"].astype("float32")
        writer.write_table(pa.Table.from_pandas(batch_df, schema=pair_schema, preserve_index=False))
        pair_count += len(batch_df)
        batch_rows = []
finally:
    writer.close()

print("도보 접근 가능 pair 수:", pair_count)
print("도보 pair 저장:", walk_pair_path)
print("도보 분석 시간(분):", round((time.time() - start_time) / 60, 2))


## 6. KTDB 대중교통 네트워크 구축

- KTDB 2025년 수도권 도로철도 통합 네트워크 사용
- KATEC 계열 좌표계를 EPSG:5179로 변환
- 경쟁권 25km + 2km buffer 내부 node를 추출하고, link는 1-hop 확장


In [ ]:
node = pd.read_csv(
    NODE_PATH,
    sep=r"\s+",
    skiprows=1,
    header=None,
    names=["record_type", "node_id", "x", "y"],
    engine="python"
).drop(columns="record_type")

link = pd.read_csv(
    LINK_PATH,
    sep=r"\s+",
    skiprows=1,
    header=None,
    engine="python"
)

transit_info = pd.read_csv(
    TRANSIT_INFO_PATH,
    sep="\t",
    encoding="cp949"
)

transit_path = pd.read_csv(
    TRANSIT_ROUTE_PATH,
    sep="\t"
)

link = link.rename(columns={
    0: "record_type",
    1: "from_node",
    2: "to_node",
    3: "length_km",
    4: "mode_code",
    5: "link_type",
    6: "lane",
    7: "capacity",
    8: "speed",
    9: "vdf",
    10: "cost",
})

link = link.drop(columns="record_type", errors="ignore")

for col in ["from_node", "to_node"]:
    link[col] = pd.to_numeric(link[col], errors="coerce").astype("Int64")
link["length_km"] = pd.to_numeric(link["length_km"], errors="coerce")
link["link_type"] = pd.to_numeric(link["link_type"], errors="coerce")
link["length_m"] = link["length_km"] * 1000

link = link[["from_node", "to_node", "length_m", "link_type"]].dropna(subset=["from_node", "to_node", "length_m"]).copy()
link["from_node"] = link["from_node"].astype(int)
link["to_node"] = link["to_node"].astype(int)

ktdb_crs = (
    "+proj=tmerc +lat_0=38 +lon_0=128 +k=0.9999 "
    "+x_0=400000 +y_0=600000 +ellps=bessel "
    "+towgs84=-146.43,507.89,681.46 +units=m +no_defs"
)

node_gdf = gpd.GeoDataFrame(
    node.copy(),
    geometry=gpd.points_from_xy(node["x"], node["y"]),
    crs=ktdb_crs
).to_crs("EPSG:5179")

print("node 구조:", node_gdf.shape)
print("link 구조:", link.shape)
print("transit_info 구조:", transit_info.shape)
print("transit_path 구조:", transit_path.shape)
print("node_id 중복:", node_gdf["node_id"].duplicated().sum())
print("link from-to 중복:", link[["from_node", "to_node"]].duplicated().sum())
print("link length 결측:", link["length_m"].isna().sum())

node_area = gpd.sjoin(
    node_gdf,
    transit_area_5179[["geometry"]],
    how="inner",
    predicate="within"
).drop(columns="index_right")

node_ids_area = set(node_area["node_id"])

link_area_hop = link[
    link["from_node"].isin(node_ids_area) |
    link["to_node"].isin(node_ids_area)
].copy()

node_ids_hop = set(link_area_hop["from_node"]) | set(link_area_hop["to_node"])
node_area_hop = node_gdf[node_gdf["node_id"].isin(node_ids_hop)].copy()

print("경쟁권 buffer 내부 node:", len(node_area))
print("1-hop 확장 node:", len(node_area_hop))
print("1-hop 확장 link:", len(link_area_hop))


## 7. 대중교통 노선 edge 생성 및 시간 cost 계산

- TransitPath에서 Line_ID별 연속 node 쌍 생성
- KTDB link 거리와 매칭하고, 정방향 미매칭은 역방향 link 거리로 보완
- 노선별 표정속도(Commercial_Speed)로 edge 이동시간 계산


In [ ]:
node_geom = node_area_hop[["node_id", "geometry"]].copy()

link_area_hop = link_area_hop.merge(
    node_geom.rename(columns={"node_id": "from_node", "geometry": "from_geometry"}),
    on="from_node",
    how="left"
)

link_area_hop = link_area_hop.merge(
    node_geom.rename(columns={"node_id": "to_node", "geometry": "to_geometry"}),
    on="to_node",
    how="left"
)

print("link 시작 geometry 결측:", link_area_hop["from_geometry"].isna().sum())
print("link 종료 geometry 결측:", link_area_hop["to_geometry"].isna().sum())

link_area_hop["geometry"] = link_area_hop.apply(
    lambda row: LineString([row["from_geometry"], row["to_geometry"]]),
    axis=1
)

transit_network = gpd.GeoDataFrame(
    link_area_hop,
    geometry="geometry",
    crs="EPSG:5179"
)

print("대중교통 link_type 분포")
print(transit_network["link_type"].value_counts().sort_index())
print("대중교통 link 거리 통계")
print(transit_network["length_m"].describe())

path_sorted = transit_path.sort_values(["Line_ID", "Seq"]).copy()
path_sorted["to_node"] = path_sorted.groupby("Line_ID")["Node_id"].shift(-1)

transit_path_edge = path_sorted[path_sorted["to_node"].notna()].copy()
transit_path_edge = transit_path_edge.rename(columns={"Node_id": "from_node"})
transit_path_edge["from_node"] = transit_path_edge["from_node"].astype(int)
transit_path_edge["to_node"] = transit_path_edge["to_node"].astype(int)

transit_path_edge = transit_path_edge[[
    "Line_ID", "Seq", "from_node", "to_node", "Station_Y/N"
]].copy()

link_cost = transit_network[["from_node", "to_node", "length_m", "link_type"]].copy()

transit_path_edge = transit_path_edge.merge(
    link_cost,
    on=["from_node", "to_node"],
    how="left"
)

node_ids_hop = set(node_area_hop["node_id"])

transit_path_target = transit_path_edge[
    transit_path_edge["from_node"].isin(node_ids_hop) &
    transit_path_edge["to_node"].isin(node_ids_hop)
].copy()

print("전체 TransitPath edge:", len(transit_path_edge))
print("경쟁권 TransitPath edge:", len(transit_path_target))
print("정방향 link 매칭 결측:", transit_path_target["length_m"].isna().sum())
print("정방향 결측 비율:", transit_path_target["length_m"].isna().mean())

link_reverse_check = link[["from_node", "to_node", "length_m", "link_type"]].rename(columns={
    "from_node": "to_node",
    "to_node": "from_node",
    "length_m": "reverse_length_m",
    "link_type": "reverse_link_type"
})

transit_path_target = transit_path_target.merge(
    link_reverse_check,
    on=["from_node", "to_node"],
    how="left"
)

forward_missing = transit_path_target["length_m"].isna()

transit_path_target["length_m"] = transit_path_target["length_m"].fillna(
    transit_path_target["reverse_length_m"]
)
transit_path_target["link_type"] = transit_path_target["link_type"].fillna(
    transit_path_target["reverse_link_type"]
)

transit_path_target["link_match_type"] = "forward"
transit_path_target.loc[
    forward_missing & transit_path_target["reverse_length_m"].notna(),
    "link_match_type"
] = "reverse_fill"
transit_path_target.loc[
    transit_path_target["length_m"].isna(),
    "link_match_type"
] = "unmatched"

print("link 매칭 유형")
print(transit_path_target["link_match_type"].value_counts())
print("최종 length_m 결측:", transit_path_target["length_m"].isna().sum())

transit_path_valid = transit_path_target[
    transit_path_target["length_m"].notna()
].copy()

transit_speed = transit_info[[
    "Line_ID", "Mode", "Type", "Name", "Commercial_Speed"
]].copy()

transit_speed["Commercial_Speed"] = pd.to_numeric(
    transit_speed["Commercial_Speed"],
    errors="coerce"
)

print("표정속도 Line_ID 중복:", transit_speed["Line_ID"].duplicated().sum())

transit_path_valid = transit_path_valid.merge(
    transit_speed,
    on="Line_ID",
    how="left"
)

transit_path_valid = transit_path_valid[
    transit_path_valid["Commercial_Speed"].notna() &
    (transit_path_valid["Commercial_Speed"] > 0)
].copy()

transit_path_valid["대중교통_edge시간"] = (
    transit_path_valid["length_m"] /
    (transit_path_valid["Commercial_Speed"] * 1000 / 60)
)

print("대중교통 edge 시간 통계")
print(transit_path_valid[["length_m", "Commercial_Speed", "대중교통_edge시간"]].describe())

transit_edge_for_graph = (
    transit_path_valid
    .sort_values("대중교통_edge시간")
    .drop_duplicates(["from_node", "to_node"], keep="first")
    .copy()
)

transit_graph = nx.DiGraph()

for row in transit_edge_for_graph.itertuples(index=False):
    transit_graph.add_edge(
        row.from_node,
        row.to_node,
        time_min=float(row.대중교통_edge시간),
        length=float(row.length_m),
        Line_ID=int(row.Line_ID),
        Mode=int(row.Mode),
        link_type=float(row.link_type) if pd.notna(row.link_type) else np.nan,
    )

print("대중교통 그래프용 edge:", len(transit_edge_for_graph))
print("대중교통 그래프 node:", transit_graph.number_of_nodes())
print("대중교통 그래프 edge:", transit_graph.number_of_edges())
print("대중교통 약연결 성분 수:", nx.number_weakly_connected_components(transit_graph))


## 8. 대중교통 node snapping 및 접근시간 산정

- 격자와 가맹점을 대중교통 그래프 node에 최근접 연결
- 격자 스냅거리는 도보 분석과 동일하게 35.35m 차감
- 보행속도는 도보 기준과 동일하게 75m/min 적용


In [ ]:
transit_graph_node_ids = list(transit_graph.nodes)
transit_node = node_area_hop[
    node_area_hop["node_id"].isin(transit_graph_node_ids)
].copy()

transit_node_nearest = transit_node[["node_id", "geometry"]].copy().rename(columns={
    "node_id": "대중교통_노드ID",
    "geometry": "대중교통_노드_geometry"
})

transit_node_nearest = gpd.GeoDataFrame(
    transit_node_nearest,
    geometry="대중교통_노드_geometry",
    crs="EPSG:5179"
)

transit_grid = grid.copy()
transit_grid["중심점"] = gpd.points_from_xy(
    transit_grid["중심점_x"],
    transit_grid["중심점_y"],
    crs="EPSG:5179"
)

transit_grid_point = gpd.GeoDataFrame(
    transit_grid.drop(columns="geometry", errors="ignore"),
    geometry="중심점",
    crs="EPSG:5179"
)

transit_store = store.copy()

transit_grid_snap = gpd.sjoin_nearest(
    transit_grid_point,
    transit_node_nearest,
    how="left",
    distance_col="대중교통_스냅거리"
).drop(columns="index_right", errors="ignore")

transit_store_snap = gpd.sjoin_nearest(
    transit_store,
    transit_node_nearest,
    how="left",
    distance_col="대중교통_스냅거리"
).drop(columns="index_right", errors="ignore")

walk_speed_m_per_min = 75
grid_snap_offset = (70.7 + 0) / 2

transit_grid_snap["대중교통_스냅거리_보정"] = (
    transit_grid_snap["대중교통_스냅거리"] - grid_snap_offset
).clip(lower=0)

transit_grid_snap["대중교통_탑승접근시간"] = (
    transit_grid_snap["대중교통_스냅거리_보정"] / walk_speed_m_per_min
)

transit_store_snap["대중교통_하차접근시간"] = (
    transit_store_snap["대중교통_스냅거리"] / walk_speed_m_per_min
)

print("격자 대중교통 node 결측:", transit_grid_snap["대중교통_노드ID"].isna().sum())
print("가맹점 대중교통 node 결측:", transit_store_snap["대중교통_노드ID"].isna().sum())
print("격자 대중교통 스냅거리 통계")
print(transit_grid_snap["대중교통_스냅거리"].describe())
print("가맹점 대중교통 스냅거리 통계")
print(transit_store_snap["대중교통_스냅거리"].describe())
print("격자 대중교통 스냅거리 500m 초과:", (transit_grid_snap["대중교통_스냅거리"] > 500).sum())
print("가맹점 대중교통 스냅거리 300m 초과:", (transit_store_snap["대중교통_스냅거리"] > 300).sum())

transit_grid_snap[
    [
        "GRID_CD",
        "대중교통_노드ID",
        "대중교통_스냅거리",
        "대중교통_스냅거리_보정",
        "대중교통_탑승접근시간",
    ]
].to_parquet(
    NETWORK_OUTPUT_PATH / "경쟁권25km_격자_대중교통노드_연결테이블.parquet",
    index=False,
    compression="zstd"
)

transit_store_snap[
    ["가맹점_ID", "대중교통_노드ID", "대중교통_스냅거리", "대중교통_하차접근시간"]
].to_parquet(
    NETWORK_OUTPUT_PATH / "경쟁권25km_가맹점_대중교통노드_연결테이블.parquet",
    index=False,
    compression="zstd"
)


## 9. 대중교통 접근 가능 pair 생성

- 수식: `접근비용 = 격자_탑승접근시간 + 대중교통_네트워크시간 + 가맹점_하차접근시간`
- 기준: 지역거점시설 차량 약 20분 기준을 대중교통 시간 cost로 변환 적용
- 실행: 가맹점 node 기준 reverse Dijkstra로 20분 이내 도달 가능한 격자 탐색


In [ ]:
transit_columns = ["미술", "공연", "스포츠관람", "관광지"]
transit_time_limit = 20

transit_grid_snap = transit_grid_snap.copy()
transit_store_snap = transit_store_snap.copy()

transit_grid_snap["대중교통_노드ID"] = transit_grid_snap["대중교통_노드ID"].astype(int)
transit_store_snap["대중교통_노드ID"] = transit_store_snap["대중교통_노드ID"].astype(int)

transit_store_target = transit_store_snap[
    transit_store_snap["중분류"].isin(transit_columns)
].copy()

node_grid_dict = defaultdict(list)
for idx, row in transit_grid_snap.iterrows():
    node_grid_dict[row["대중교통_노드ID"]].append(idx)

node_store_dict = defaultdict(list)
for idx, row in transit_store_target.iterrows():
    node_store_dict[row["대중교통_노드ID"]].append(idx)

grid_info = transit_grid_snap.to_dict("index")
store_info = transit_store_target.to_dict("index")

print("대중교통 대상 가맹점 수:", len(transit_store_target))
print("대중교통 대상 가맹점 node 수:", len(node_store_dict))
print("격자가 붙은 대중교통 node 수:", len(node_grid_dict))
print(transit_store_target["중분류"].value_counts())

transit_pair_path = NETWORK_OUTPUT_PATH / "경쟁권25km_대중교통접근성.parquet"
if transit_pair_path.exists():
    transit_pair_path.unlink()

pair_schema = pa.schema([
    ("GRID_CD", pa.string()),
    ("가맹점_ID", pa.string()),
    ("접근수단", pa.string()),
    ("접근비용", pa.float32()),
])

writer = pq.ParquetWriter(transit_pair_path, pair_schema, compression="zstd")

transit_graph_reverse = transit_graph.reverse(copy=True)
min_grid_access_time = transit_grid_snap["대중교통_탑승접근시간"].min()

batch_rows = []
batch_size = 500_000
pair_count = 0
start_time = time.time()
total_store_node_count = len(node_store_dict)

try:
    for n, (store_node, store_idx_list) in enumerate(node_store_dict.items(), start=1):
        if store_node not in transit_graph_reverse:
            continue

        min_store_access_time = min(
            store_info[store_idx]["대중교통_하차접근시간"]
            for store_idx in store_idx_list
        )

        network_cutoff = transit_time_limit - min_grid_access_time - min_store_access_time
        if network_cutoff <= 0:
            continue

        lengths = nx.single_source_dijkstra_path_length(
            transit_graph_reverse,
            store_node,
            cutoff=network_cutoff,
            weight="time_min"
        )

        for grid_node, network_time in lengths.items():
            grid_idx_list = node_grid_dict.get(grid_node, [])
            if len(grid_idx_list) == 0:
                continue

            for grid_idx in grid_idx_list:
                grid_row = grid_info[grid_idx]
                grid_access_time = grid_row["대중교통_탑승접근시간"]

                for store_idx in store_idx_list:
                    store_row = store_info[store_idx]
                    store_access_time = store_row["대중교통_하차접근시간"]

                    final_time = (
                        grid_access_time
                        + network_time
                        + store_access_time
                    )

                    if final_time <= transit_time_limit:
                        batch_rows.append((
                            str(grid_row["GRID_CD"]),
                            str(store_row["가맹점_ID"]),
                            "대중교통",
                            float(final_time),
                        ))

        if len(batch_rows) >= batch_size:
            batch_df = pd.DataFrame(
                batch_rows,
                columns=["GRID_CD", "가맹점_ID", "접근수단", "접근비용"]
            )
            batch_df["접근비용"] = batch_df["접근비용"].astype("float32")
            writer.write_table(pa.Table.from_pandas(batch_df, schema=pair_schema, preserve_index=False))
            pair_count += len(batch_df)
            batch_rows = []

        if n % 50 == 0:
            elapsed_min = (time.time() - start_time) / 60
            print(f"{n:,} / {total_store_node_count:,}개 대중교통 가맹점 node 처리 | pair {pair_count:,}+ | {elapsed_min:.1f}분")

    if len(batch_rows) > 0:
        batch_df = pd.DataFrame(
            batch_rows,
            columns=["GRID_CD", "가맹점_ID", "접근수단", "접근비용"]
        )
        batch_df["접근비용"] = batch_df["접근비용"].astype("float32")
        writer.write_table(pa.Table.from_pandas(batch_df, schema=pair_schema, preserve_index=False))
        pair_count += len(batch_df)
        batch_rows = []
finally:
    writer.close()

print("대중교통 접근 가능 pair 수:", pair_count)
print("대중교통 pair 저장:", transit_pair_path)
print("대중교통 분석 시간(분):", round((time.time() - start_time) / 60, 2))


## 10. 저장 산출물 검토

- 통합 접근성 테이블은 중복 저장하지 않음
- 접근성 분석 시 `분석격자`, `분석가맹점`, `도보접근성`, `대중교통접근성`을 key 기준으로 결합
- 각 pair 테이블의 접근비용은 기존과 동일하게 접근수단별 단위를 가짐


In [ ]:
saved_files = sorted(NETWORK_OUTPUT_PATH.glob("*"))

print("저장 파일")
for path in saved_files:
    size_mb = path.stat().st_size / 1024 / 1024
    print(f"{path.name}: {size_mb:,.1f} MB")

for label, path in [
    ("도보", NETWORK_OUTPUT_PATH / "경쟁권25km_도보접근성.parquet"),
    ("대중교통", NETWORK_OUTPUT_PATH / "경쟁권25km_대중교통접근성.parquet"),
]:
    if path.exists() and path.stat().st_size > 0:
        df_head = pd.read_parquet(path, columns=["GRID_CD", "가맹점_ID", "접근수단", "접근비용"])
        print(f"\n{label} pair 구조:", df_head.shape)
        print(f"{label} 접근비용 통계")
        print(df_head["접근비용"].describe())
        print(f"{label} 연결 격자 수:", df_head["GRID_CD"].nunique())
        print(f"{label} 연결 가맹점 수:", df_head["가맹점_ID"].nunique())
        print(f"{label} 중복 pair:", df_head[["GRID_CD", "가맹점_ID", "접근수단"]].duplicated().sum())
        display(df_head.head())
    else:
        print(f"{label} pair 파일 없음 또는 빈 파일")
